In [1]:
import random
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import style
import matplotlib as mpl
import scanpy as sc
import anndata
import os.path as osp
from sklearn.metrics import adjusted_rand_score

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
import sys
from pathlib import Path

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# GPU hardware acceleration
import torch 
device='cuda' if torch.cuda.is_available() else 'cpu'
%load_ext autoreload
%autoreload 2
import time
import spateo as st
import pyvista as pv
import pickle as pkl

Merfish mouse brain data

In [ ]:

spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# set random seeds
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]

in_ = '/home/data3/yuling/xenium_region2_5599/merfish_mouse_brain'
slice_ids_o = range(33)
sliceID =[[str(x) for x in slice_ids_o]]
end_no = 33
slices = [] #this is the original data
slice_ids = sliceID[0]
for i in range(len(slice_ids)):
    slice1 = sc.read(osp.join(in_, 'merfish_mouse_brain_slice'+slice_ids[i]+'.h5ad'))
    slice1.obs['original_clusters']=slice1.obs['spa_cluster'].astype('category')
    slice1.obsm['spatial'] = slice1.obsm['spatial'].to_numpy()[:, :2]
    slice1.obs['Z']=i
    slices.append(slice1)
    
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo_merfish', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)

    np.random.seed(i)
    torch.manual_seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i)  # If using multi-GPU
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)

    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=aligned_adata.obs_names.to_list())
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_merfish.csv'))

simulated data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'

slice_ids =[151673,100,80,60,40,20]
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]

    # load data and normalize
slices=[]
slice1=sc.read_h5ad('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673original_spotIndex.h5')
slice1.layers["counts"] = slice1.X.copy()
sc.pp.normalize_total(slice1)
sc.pp.log1p(slice1)
slice1.obs['slice_id']=151673
slice1.obs['Z']=0
slice1.obs_names_make_unique()
slices.append(slice1)
for index,s in enumerate(slice_ids[1:]):
    slice1=sc.read_h5ad(osp.join('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673_overlap='+str(s)+'__pseudocount_0_spotIndex.h5'))
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['slice_id']=s
    slice1.obs['Z']=index+1
        
    slice1.obs_names_make_unique()
    slices.append(slice1) 
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/simu_spateo', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)

    np.random.seed(i)

    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)

    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=aligned_adata.obs_names.to_list())
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_simu.csv')) 


Mouse Hypothalamus data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
slice_ids = ['-0.04', '-0.09',  '-0.14', '-0.19' , '-0.24']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
# load data and normalize
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_mHypothalamus(section_id=section_ids)
    slice1.var_names_make_unique()
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo/mHy', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=aligned_adata.obs_names.to_list())
    
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_mHy.csv'))



DLPFC data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# ['151669', '151670',  '151671', '151672']
# ['151673', '151674',  '151675', '151676']
slice_ids = ['151507', '151508',  '151509', '151510']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]

for i in seeds_to_run:
    # load data and normalize
    slices=[]
    for j,section_ids in enumerate(slice_ids):
        slice1 = load_DLPFC(section_id=section_ids)
        slice1.layers["counts"] = slice1.X.copy()
        sc.pp.normalize_total(slice1)
        sc.pp.log1p(slice1)
        slice1.obs['Z']=j
        slice1.obs['slice_id']=section_ids
        
        slices.append(slice1)
    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo/DLPFC_4/151507_151510', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)

    np.random.seed(i)
    torch.manual_seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i)  # If using multi-GPU
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')

        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
#### save coordinates
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=aligned_adata.obs_names.to_list())

    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_151507_151510_1.csv'))


MOSTA embryo data

In [ ]:
# load Mosta Embryo data
def load_Mostaembryo(root_dir='/home/data3/yuling/xenium_region2_5599/embryo', section_id='11.5_E1S1.MOSTA'):
    #23clusters in 12.5, 19 clusters in 11.5: [Cavity, Meninges, Surface ectoderm, Blood vessel, Connective tissue, Dermomyotome, Brain, Jaw and tooth, GI tract, Dorsal root ganglion, Inner ear, Head mesenchyme, Lung primordium, Sclerotome, Branchial arch, Urogenital ridge, Heart, Liver, Spinal cord]
    # n_obs × n_vars = 51365 × 27810
    ad = sc.read(osp.join(root_dir, section_id+'.h5ad'))
    ad.X = ad.layers['count']
    ad.var_names_make_unique()
    keep_bcs = ad.obs.dropna().index
    ad = ad[keep_bcs].copy()
    ad.obs['original_clusters'] = ad.obs['annotation']
    return ad


spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
slice_ids = ['E11.5_E1S1.MOSTA', 'E11.5_E1S2.MOSTA',  'E11.5_E1S3.MOSTA', 'E11.5_E1S4.MOSTA']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
# load data and normalize
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_Mostaembryo(section_id=section_ids)
        #slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)

for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join('/home/data3/yuling/xenium_region2_5599/spateo_align/mosta_embryo', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    for cur_idx in range(1,len(slice_ids)):
        index = slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path = osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    
    aligned_adata = anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=aligned_adata.obs_names.to_list())
    
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_mosta_embryo.csv'))


In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
slice_ids = ['E9.5_E1S1.MOSTA','E10.5_E1S1.MOSTA','E11.5_E1S1.MOSTA', 'E12.5_E1S1.MOSTA',  'E13.5_E1S1.MOSTA', 'E14.5_E1S1.MOSTA','E15.5_E1S1.MOSTA','E16.5_E1S1.MOSTA']
seeds_to_run=[2024,1,2,3,4,10,11,12,13,14]
# dataset is not normalized
# load data and normalize
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_Mostaembryo(section_id=section_ids)
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)

path_ = '/home/data3/yuling/xenium_region2_5599/spateo_align/mosta_embryo_8'
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join(path_, str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis, align_time, time_3d = st.align.morpho_align(
            models=slices,
            device=device,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True
            )
    # Save alignment pi

    dataset = []
    for cur_idx in range(1,len(slice_ids)):
        index = slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        dataset.append(index)
        map_path = osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)

    aligned_adata = anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_mosta_embryo_8.csv'))

Human metastatic lymph node (19 slices)

In [ ]:
slice_ids = ["2", "3", "4", "5", "6", "7", "9", "11", "17", "18", "19", "23", "24", "25", "26", "28", "33", "34", "36"]
def load_HMlymphNode(root_dir = '/home/data/yulingzhu/GSE251926_metastatic_lymph_node_3d.h5ad', section_id =  "1"):
    adataT = sc.read_h5ad(root_dir)
    section_id = int(section_id)  # Convert section_id to integer
    slice1 = adataT[adataT.obs['n_section'] == section_id]
    slice1.obs['original_clusters'] = slice1.obs['annotation']
    return slice1

slices=[]
for index,i in enumerate(slice_ids):
    ad = load_HMlymphNode(section_id=i)
    ad.obs['Z'] = index
    ad = anndata.AnnData(
        X = ad.X.copy(),
        obs = ad.obs[['annotation','original_clusters','Z']].copy(),
        var = ad.var[['total_counts']].copy(),
        obsm = ad.obsm.copy()
    )
    ad.obs['slice_id'] = i
    ad.layers["counts"] = ad.X.copy()
    slices.append(ad) 

spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# [2024,1,2,3,4,10,11,12,13,14]
seeds_to_run = [2024,1,2,3,4,10,11,12,13,14] 

for i in seeds_to_run:
    fold_path=osp.join('/home/data/yulingzhu/spateo/lymph', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 
    aligned_slices, pis = st.align.morpho_align(
            models=slices,
            device=device,
             
            partial_robust_level=10,
        # nn_init=False,
            lambdaVF=1e1,
            beta=1e-1,
            sigma2_init_scale=1,
          
            sparse_calculation_mode=False,
            use_chunk=False,
            chunk_capacity=2,
            rep_layer=['X'],
            rep_field=['layer'],
            dissimilarity=['kl'],
            # If computed PCA, you can use:
            # rep_layer=['X_pca'],
            # rep_field=['obsm'],
            # dissimilarity=['cos'],
            probability_type=['gauss'],
            nn_init=True,
            nn_init_weight=1,
            max_iter=1000,
            batch_size=4000,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True,
            )
    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=aligned_adata.obs_names.to_list())
    
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_lymph.csv'))


Hemibrain (129 slices)

In [ ]:
# load data
data_dir = Path('/home/users/yulingzhu/3d_project/mHemibrain/4point2millionCell')
output_dir = data_dir / 'split_sections'
output_file = output_dir / 'section_labels_ccf.csv'
# Read the CSV file
section_labels_df = pd.read_csv(output_file)
slice_ids =[]
# Extract the labels as a list
section_labels = section_labels_df['brain_section_label'].tolist()
slice_labels = []
for i,section in enumerate(section_labels):
    section_file = output_dir / f"{section}_ccf.h5ad"
    slice_ids.append(section)
    section_data = anndata.read_h5ad(section_file)  # Load the section file
    section_data.obs['neurotransmitter'] = section_data.obs['neurotransmitter'].cat.add_categories("NaN")
        # Replace NaN values with the string "NaN"
    section_data.obs['original_clusters'] = section_data.obs['neurotransmitter'].fillna("NaN").astype(str)
    section_data.obs['Z']=i 
    section_data.obs['slice_id']=section
    sc.pp.normalize_total(section_data)
    sc.pp.log1p(section_data)
    slice_labels.append(section_data) 
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# [2024,1,2,3,4,10,11,12,13,14]
seeds_to_run = [2024,1,2,3,4,10,11,12,13,14]
 
for i in seeds_to_run:
    # create folder for each seed
    fold_path=osp.join('/home/data/yulingzhu/spateo/Hemibrain', str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    
    np.random.seed(i)
    #random.seed(i) 
    torch.manual_seed(i) 
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i) 

    aligned_slices, pis = st.align.morpho_align(
            models=slice_labels,
            device=device,
            mode="SN-S",
            max_iter=200,
            partial_robust_level=50,
            beta=1e-5,
            lambdaVF=1e5,
            SVI_mode=True,
            nn_init=True,
            iter_key_added=None,
            verbose=False,
            spatial_key=spatial_key,
            key_added=key_added,
            return_mapping=True,
            fold_path=fold_path,
            seed = i
            )
    # Assuming morpho_model is an instance of Morpho_pairwise

    dataset=[]

    for cur_idx in range(1,len(slice_ids)):
        index=slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path=osp.join(fold_path, index+'_AlignmentPi.pkl')
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    aligned_adata=anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D']=np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=aligned_adata.obs_names.to_list())
    df['original_clusters']=aligned_adata.obs['original_clusters'].values
    df['slice']=aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_Hemibrain.csv'))

HBC data

In [ ]:
spatial_key = 'spatial'
cluster_key = 'original_clusters'
key_added = 'align_spatial'
# set random seeds
seeds_to_run = [2024,1,2,3,4,10,11,12,13,14]
path_ = '/home/data3/yuling/xenium_region2_5599/spateo_align/HBC'
slice_ids = ['xenium_rep1', 'xenium_rep2' ]
slices=[]
for index,i in enumerate(slice_ids):
    adata_1 = sc.read_h5ad(osp.join('/home/data3/yuling/xenium_region2_5599/HBC_labels', i+'_io_aligned.h5'))
    adata_1.obs['original_clusters'] = adata_1.obs['celltype_minor']
    adata_1.obs['Z']=str(index)
    adata_1.obs['slice_id']=i
    slices.append(adata_1)
for i in seeds_to_run:

    # create folder for each seed
    fold_path=osp.join(path_, str(i))
    if not os.path.exists(fold_path):
        os.makedirs(fold_path)
    np.random.seed(i)
    torch.manual_seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(i)
        torch.cuda.manual_seed_all(i)  # If using multi-GPU
    aligned_slices, pis, align_time, time_3d = st.align.morpho_align(
            models = slices,
            device = device,
            verbose = False,
            spatial_key = spatial_key,
            key_added = key_added,
            return_mapping = True
            )

    dataset = []
    for cur_idx in range(1,len(slice_ids)):
        index = slice_ids[cur_idx-1]+'_'+slice_ids[cur_idx]
        map_path = osp.join(fold_path, index+'_AlignmentPi.pkl')
        dataset.append(index)
        with open(map_path,'wb') as f:
            pkl.dump(pis[cur_idx-1], f)
    
    aligned_adata = anndata.concat(aligned_slices)
    aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['align_spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
    df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    
    df['original_clusters'] = aligned_adata.obs['original_clusters'].values
    df['slice'] = aligned_adata.obs['slice_id'].values
    df.to_csv(osp.join(fold_path,'3d_coord_HBC.csv'))
 